## （可選）對照 Phase 1 Job

Workbench 與 Job 共用同一 PVC。請優先用 **OpenShift Console**：

1. Console → 專案 `corrdiff-poc` → **Workloads → Jobs** → 查看 Logs  
2. 若要重跑：Console → **+ → Import YAML**，貼本包 `k8s/inference-job.yaml`（勿多人同時搶 GPU）

講師若使用本機 `oc`（學員可跳過）：

```bash
oc delete job corrdiff-inference -n corrdiff-poc --ignore-not-found
oc apply -f k8s/inference-job.yaml
oc logs -f job/corrdiff-inference -n corrdiff-poc
```


In [ ]:
import os, sys, subprocess
import torch
import netCDF4 as nc
import numpy as np

SHOME = "/mnt/corrdiff"
DATE = "20260707"
FC, OP, CD = "ec46day", "opv1", "corrdiff_v1"
CONFIG = f"{SHOME}/workdir/{DATE}/config.yaml"
INPUT_NC = f"{SHOME}/workdir/{DATE}/CorrdiffInput_EC_RAW_{DATE}.nc"
OUTPUT_NC = f"{SHOME}/dtg/EC_S2S_AIPP/{DATE}/CorrdiffOutput_EC_RAW_{DATE}.nc"

os.environ["HOME"] = "/tmp"
os.environ["LOCAL_CACHE"] = "/tmp/.cache/modulus"
os.makedirs(os.environ["LOCAL_CACHE"], exist_ok=True)

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import modulus
from modulus import Module
print("modulus OK")

In [ ]:
# 若 config 含 HPC 路徑 (/nwpr/...)，用 SHOME=/mnt/corrdiff 重生
from pathlib import Path
import subprocess

cfg_text = Path(CONFIG).read_text() if Path(CONFIG).exists() else ""
if "/nwpr/" in cfg_text or "etc_dir" not in cfg_text or "/mnt/corrdiff/etc" not in cfg_text:
    print("Regenerating config.yaml with SHOME=/mnt/corrdiff ...")
    subprocess.run(
        [
            "python3", f"{SHOME}/bin/config_gen.py",
            "-i", f"{SHOME}/config/gen_config.yaml",
            "-o", CONFIG,
            "-v", f"dtg={DATE}",
            "-v", f"SHOME={SHOME}",
        ],
        check=True,
    )
else:
    print("config.yaml already uses /mnt/corrdiff")

for line in Path(CONFIG).read_text().splitlines():
    if "etc_dir" in line or "out_dir" in line:
        print(line)
assert "/mnt/corrdiff/etc" in Path(CONFIG).read_text(), "etc_dir must be under /mnt/corrdiff"
assert "/nwpr/" not in Path(CONFIG).read_text(), "HPC /nwpr paths still present"


In [ ]:
from pathlib import Path

from pathlib import Path

checks = {
    "config": CONFIG,
    "input": INPUT_NC,
    "UNet model": f"{SHOME}/etc/UNet.Under850_v1.mdlus",
    "EDM model": f"{SHOME}/etc/EDMPrecondSR.Under850_v1.mdlus",
    "stats": f"{SHOME}/etc/ERA_TREAD_19910101_20231231_normalize_parameter.txt",
    "grid": f"{SHOME}/etc/wrf_208x208_grid_coords.nc",
    "inference script": f"{SHOME}/bin/inference_v1.py",
    "lib (via bin/lib)": f"{SHOME}/bin/lib/corrdiff_inference.py",
    "lib (via /lib)": f"{SHOME}/lib/corrdiff_inference.py",
}

for name, path in checks.items():
    p = Path(path)
    status = "OK" if p.exists() else "MISSING"
    size = f" ({p.stat().st_size/1024/1024:.1f} MB)" if p.is_file() else ""
    print(f"[{status}] {name}: {path}{size}")

# bin/lib 常是 symlink → ../lib；缺 lib/ 會出現 ModuleNotFoundError: lib
lib_ok = Path(f"{SHOME}/bin/lib/corrdiff_inference.py").exists() or Path(
    f"{SHOME}/lib/corrdiff_inference.py"
).exists()
assert lib_ok, (
    "缺少 lib/corrdiff_inference.py。請講師重跑更新後的 seed-data.sh，"
    "或手動：oc cp corrdiff_for_ocp/lib/. <ns>/<seed-pod>:/mnt/corrdiff/lib/"
)

In [ ]:
ds = nc.Dataset(INPUT_NC)
print("Input dimensions:", {k: len(ds.dimensions[k]) for k in ds.dimensions})
fc = ds.variables["fc"]
print("fc shape:", fc.shape, "dtype:", fc.dtype)
sample = np.array(fc[0, 0, :5, :5])
print("sample corner values:\n", sample)
ds.close()

## 執行完整推論

以下呼叫與 Phase 1 `corrdiff-inference` Job 相同的 `inference_v1.py`。
預估需數分鐘（45 lead days × GPU 推論）。

In [ ]:
import os
os.makedirs(f"{SHOME}/dtg/EC_S2S_AIPP/{DATE}", exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = f"{SHOME}/bin:" + env.get("PYTHONPATH", "")
env["HOME"] = "/tmp"
env["LOCAL_CACHE"] = "/tmp/.cache/modulus"

cmd = [
    sys.executable, f"{SHOME}/bin/inference_v1.py",
    DATE, CONFIG, FC, OP, CD,
]
print(" ".join(cmd))
proc = subprocess.run(cmd, cwd=f"{SHOME}/bin", env=env, capture_output=False)
print("exit code:", proc.returncode)
assert proc.returncode == 0, "inference failed"

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

out = Path(OUTPUT_NC)
assert out.exists(), f"missing output: {OUTPUT_NC}"
print(out, f"{out.stat().st_size/1024/1024:.1f} MB")

ds = nc.Dataset(OUTPUT_NC)
print("Output dimensions:", {k: len(ds.dimensions[k]) for k in ds.dimensions})
fc = ds.variables["fc"]
print("fc shape:", fc.shape)

# 簡單視覺化：case=0, lead=0, var=0 (temperature_2m)
field = np.array(fc[0, 0, 0, :, :])
plt.figure(figsize=(6, 5))
plt.imshow(field, origin="lower")
plt.title("CorrDiff output — lead day 1, var 0")
plt.colorbar()
plt.show()
ds.close()

## （可選）從 Notebook 提交 Phase 1 Job

若要在 UI 外重跑批次推論，可在 Terminal 執行：

```bash
oc delete job corrdiff-inference -n corrdiff-poc --ignore-not-found
# Prefer Console → Jobs → Logs, or Import YAML from this pack:
# k8s/inference-job.yaml
# Instructor with oc:
# oc apply -f k8s/inference-job.yaml
oc logs -f job/corrdiff-inference -n corrdiff-poc
```